In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

In [ ]:
!git clone https://github.com/yszhang95/tred.git -b 2x2_scaleq_playground

In [ ]:
%cd tred
!uv pip install -e .

In [ ]:
import tred
from tred.blocking import Block
from tred.chunking import accumulate
from tred.sparse import chunkify, SGrid

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

In [ ]:
import torch
from torch import nn

# ---------- FFT-based 3D linear convolution ----------
def fft_convolve3d(x, k):
    """
    Linear 3D convolution via FFT.

    x: (1, D, H, W)
    k: (Kd, Kh, Kw) or broadcastable to x (no channel dim assumed)
    mode: 'same' -> output (1, D, H, W), 'full' -> (1, D+Kd-1, H+Kh-1, W+Kw-1)
    """
    assert x.dim() == 4 and x.shape[0] == 1, "x must be (1, D, H, W)"
    D, H, W = x.shape[1:]
    Kd, Kh, Kw = k.shape[-3:]

    # Full linear conv size
    Df, Hf, Wf = D + Kd - 1, H + Kh - 1, W + Kw - 1

    # Next pow2 sizes for speed
    # def next_pow2(n):
    #     return 1 << (n - 1).bit_length()
    # nD, nH, nW = next_pow2(Df), next_pow2(Hf), next_pow2(Wf)
    nD, nH, nW = Df, Hf, Wf

    # FFTs
    Xf = torch.fft.rfftn(x, s=(nD, nH, nW), dim=(1,2,3))
    Kf = torch.fft.rfftn(k, s=(nD, nH, nW), dim=(0,1,2))
    Yf = Xf * Kf  # broadcast over batch

    y_full = torch.fft.irfftn(Yf, s=(nD, nH, nW), dim=(1,2,3))[:, :Df, :Hf, :]

    y_center = y_full[:, Kd//2:D+Kd//2, Kh//2:H+Kh//2, :]

    return y_center

# ---------- First difference along the last axis ----------
def diff_3d(x):
    """First difference along the last axis (time)."""
    dx = x[:, 1:, :, :] - x[:, :-1, :, :]
    dy = x[:, :, 1:, :] - x[:, :, :-1, :]
    dz = x[:, :, :, 1:] - x[:, :, :, :-1]
    return dx, dy, dz

# ---------- Apply a matrix along the last axis ----------
def apply_mat_last_axis(t, M):
    """
    t: (..., W) volume or tensor
    M: (Mout, W) matrix acting on the last axis
    returns: (..., Mout)
    """
    *lead, W = t.shape
    t2 = t.reshape(-1, W)                # (N, W)
    out = t2 @ M                       # (N, Mout)
    return out.reshape(*lead, M.shape[0])

# ---------- Objective module ----------
class Deconv3DObjective(nn.Module):
    def __init__(self, Ae, A0, K, Mask, lam_l1=0., lam_dx=0., lam_a0=0., conv_mode="same"):
        super().__init__()
        # store as buffers so they follow .to(device)
        self.register_buffer("Ae", Ae)    # (M, W')
        self.register_buffer("A0", A0)    # (P, W')
        self.register_buffer("K",  K)     # (Kd, Kh, Kw)
        self.lam_l1 = lam_l1
        self.lam_dx = lam_dx
        self.lam_a0 = lam_a0
        self.register_buffer("Mask", Mask)
        self.conv_mode = conv_mode
        self.relu = nn.ReLU()

    def forward(self, Z, Y):
        """
        Z: (1, D, H, W) unconstrained
        Y: (1, D', H', M) with M == Ae.shape[0]
        """
        X = self.relu(Z) * self.Mask                             # enforce X >= 0
        KX = fft_convolve3d(X, self.K)   # (1, D', H', W')
        # print('Input to FFT', X.shape, 'Kernel to FFT', self.K.shape,
        #       'KX after FFT', KX.shape, self.Ae.shape, self.A0.shape)

        # Apply Ae and A0 along the last axis independently at each (d,h) location
        # AeKX = apply_mat_last_axis(KX, self.Ae)      # (1, D', H', M)
        # A0KX = apply_mat_last_axis(KX, self.A0)      # (1, D', H', P)
        # print(self.Ae.dtype, KX.dtype)
        AeKX = self.Ae.to(KX.dtype) @ KX.unsqueeze(-1)
        AeKX = AeKX.squeeze(-1)
        A0KX = self.A0.to(KX.dtype) @ KX.unsqueeze(-1)
        A0KX = A0KX.squeeze(-1)

        # Data fidelity
        data_term = torch.sum((AeKX - Y) ** 2)

        # Regularizers
        l1_term = self.lam_l1 * torch.sum(X)         # X >= 0 -> |X| = X
        dx, dy, dz = diff_3d(X)
        dx_term = self.lam_dx * torch.sqrt((dx**2).sum() + (dy**2).sum() + (dz**2).sum())
        a0_term = self.lam_a0 * torch.linalg.vector_norm(A0KX)

        loss = data_term + l1_term + dx_term + a0_term
        return loss, {"X": X, "KX": KX, "AeKX": AeKX, "A0KX": A0KX,
                      "data": data_term, "l1": l1_term, "dx": dx_term, "a0": a0_term}

# ---------- Solver ----------
def solve_nonnegative_3d(
    Ae, A0, K, Y, Mask,
    lam_l1=0., lam_dx=0., lam_a0=0.,
    steps=1000, lr=1e-2, Z0=None, device=None, progress_every=100
):
    """
    Ae: (1, D', H', M, W')  matrix on last axis
    A0: (1, D', H', P, W')  matrix on last axis (cumulative integrator)
    K : (Kd, Kh, Kw)
    Y : (1, D', H', M)  matches Ae applied to last axis
    Mask : Mask on X
    Returns: M @ X_hat (1, D, H, W) and a small info dict
    """
    dev = device or (Ae.device if isinstance(Ae, torch.Tensor) else "cpu")
    Ae, A0, K, Y, Mask = Ae.to(dev), A0.to(dev), K.to(dev), Y.to(dev), Mask.to(dev)

    # Infer input spatial size if using "same": W' == input W
    # We'll initialize Z to match the *pre-conv* size implied by 'same' mode.

    BAe, Dp, Hp, M, Wp = Ae.shape
    assert BAe == 1 and Dp == Y.shape[1] and Hp == Y.shape[2] and M == Y.shape[3], f'Ae.shape {Ae.shape} does not match on Y.shape {Y.shape}'
    B, _, _, _ = Y.shape
    assert B == 1, 'batch size != 1'

    # Input W must equal Ae's width W' under 'same' mode
    Kd, Kh, Kw = K.shape
    W = Wp - Kw + 1
    D = Dp
    H = Hp
    assert all(d1 == d2 for d1, d2 in zip(Mask.shape[1:], (D, H, W))), f"Mask.shape[1:] {Mask.shape[1:]}, (D,H,W) {(D,H,W)}"
    init_shape = (1, D, H, W)

    # Initialize Z (unconstrained)
    if Z0 is None:
        Z = torch.full(init_shape, -2.0, device=dev, requires_grad=True)
    else:
        Z = Z0.clone().detach().to(dev).requires_grad_(True)

    obj = Deconv3DObjective(Ae, A0, K, Mask, lam_l1=lam_l1, lam_dx=lam_dx, lam_a0=lam_a0).to(dev)
    opt = torch.optim.Adam([Z], lr=lr)

    history = []
    for t in range(1, steps + 1):
        opt.zero_grad()
        loss, parts = obj(Z, Y)
        loss.backward()
        opt.step()

        if progress_every and (t % progress_every == 0 or t == 1 or t == steps):
            with torch.no_grad():
                history.append({
                    "step": t,
                    "loss": float(loss),
                    "data": float(parts["data"]),
                    "l1": float(parts["l1"]),
                    "dx": float(parts["dx"]),
                    "a0": float(parts["a0"]),
                })

    with torch.no_grad():
        X_hat = obj.relu(Z)*Mask
        KX    = fft_convolve3d(X_hat, K)
        AeKX  = apply_mat_last_axis(KX, Ae)

    return X_hat, {"KX": KX, "AeKX": AeKX, "history": history}


In [ ]:
fres = np.load("/content/drive/MyDrive/Colab Notebooks/response_v2a_distance_10p431cm_binsize_0p04434cm_tick0p05us.npy")

In [ ]:
fres3x3 = np.empty((3,3, 401)) # rebin to 50ns
# center at [1,1]
# [[0,0],[0,1], [0,2],
#  [1,0], [1,1], [1,2],
#  [2,0], [2,1], [2,2]]
fres3x3[1,1] = np.mean(fres[:5,:5], axis=(0,1))[600-2::2]
fres3x3[2,1] = np.mean(fres[:5,5:10], axis=(0,1))[600-2::2]
fres3x3[1,2] = np.mean(fres[5:10,:5], axis=(0,1))[600-2::2]
fres3x3[2,2] = np.mean(fres[5:10,5:10], axis=(0,1))[600-2::2]
fres3x3[0,0] = fres3x3[2,2]
fres3x3[0,1] = fres3x3[2,1]
fres3x3[1,0] = fres3x3[1,2]
fres3x3[0,2] = fres3x3[2,2]
fres3x3[2,0] = fres3x3[2,2]
fres3x3 = fres3x3/np.sum(fres3x3[1,1])

In [ ]:
for i in range(3):
    for j in range(3):
        plt.plot(fres3x3[i,j], label=f"[{i},{j}]")
plt.legend()

In [ ]:
finput = np.load("/content/drive/MyDrive/Colab Notebooks/single_track_full_fr_20250924.npz")

In [ ]:
finput.files

In [ ]:
finput['hits_tpc2_batch11'], finput['one_tick'], finput['time_spacing']

In [ ]:
effql = torch.tensor(finput['effq_tpc2_batch11_location'])
effq = torch.tensor(finput['effq_tpc2_batch11'][:,3])
effqb = Block(location=effql, data=effq.unsqueeze(1).unsqueeze(1).unsqueeze(1))

In [ ]:
def coarse_chunk(block, shp_tensor):
    block =  accumulate(chunkify(block, shp_tensor))
    block = Block(data=block.data, location=block.location)
    return block

def block_on_sgrid(block, shp_tensor):
    block = coarse_chunk(block, shp_tensor)
    location = SGrid(shp_tensor).spoint(block.location)
    return Block(location=location, data=block.data.sum(dim=(1,2,3), keepdim=True))

In [ ]:
# everything on 100ns basis

effqb_sgrid = block_on_sgrid(effqb, (1,1,2))
coarse_effqb_sgrid = coarse_chunk(effqb_sgrid, (1,1,25))
torch.min(effqb.location[:,-1]), torch.max(effqb.location[:,-1]), torch.min(coarse_effqb_sgrid.location[:,-1]), torch.max(coarse_effqb_sgrid.location[:,-1])

In [ ]:
import plotly.graph_objects as go

In [ ]:
fig = go.Figure(data=go.Scatter3d(x=effqb_sgrid.location[:,0],
                                  y=effqb_sgrid.location[:,1],
                                  z=effqb_sgrid.location[:,2],
                                  mode="markers", marker=dict(size=1, color=effqb_sgrid.data.squeeze())))
fig.show()

In [ ]:
# convert hit from 50ns basis to 100ns basis
hits_loc = finput['hits_tpc2_batch11_location'][:,[0,1,3]]
hits_q = finput['hits_tpc2_batch11'][:,3]

In [ ]:
finput['hits_tpc2_batch11_location'].shape

In [ ]:
hb = Block(location=torch.tensor(hits_loc), data=torch.tensor(hits_q).unsqueeze(1).unsqueeze(1).unsqueeze(1))
hb_sgrid = block_on_sgrid(hb, (1,1,2))

In [ ]:
fig = go.Figure(data=go.Scatter3d(x=hb_sgrid.location[:,0],
                                  y=hb_sgrid.location[:,1],
                                  z=hb_sgrid.location[:,2],
                                  mode="markers", marker=dict(size=1, color=hb_sgrid.data.squeeze())))
fig.show()

In [ ]:
# generate masks
support_trange = 25
pre_n = 2
post_n = 2

tshift = np.argmax(fres3x3[1,1])

tmin = torch.min(hb_sgrid.location[:,-1] - tshift - pre_n * support_trange) // support_trange * support_trange
tmax = torch.max(hb_sgrid.location[:,-1] - tshift + post_n * support_trange) // support_trange * support_trange
print(torch.max(effqb_sgrid.location[:,-1]) + 300 )
print(torch.min(effqb_sgrid.location[:,-1]) + 300 )

print(
    'tmin', tmin, 'tmax', tmax, 'hit min', torch.min(hb_sgrid.location[:,-1]), 'tshift', tshift, torch.min(hb_sgrid.location[:,-1])-tshift)

xmin = torch.min(hb_sgrid.location[:,0])
xmax = torch.max(hb_sgrid.location[:,0])+1
ymin = torch.min(hb_sgrid.location[:,1])
ymax = torch.max(hb_sgrid.location[:,1])+1
print(xmax-xmin, ymax-ymin, tmax-tmin)

hb_sgrid_loc_on_min = Block(location=hb_sgrid.location - torch.tensor([xmin, ymin, tmin]), data=hb_sgrid.data)
hb_sgrid_loc_on_min_tshift = Block(location=hb_sgrid.location - torch.tensor([xmin, ymin, tmin+tshift]), data=hb_sgrid.data)
print(torch.min(hb_sgrid_loc_on_min_tshift.location[:,-1]), torch.max(hb_sgrid_loc_on_min_tshift.location[:,-1]))

mhd = torch.zeros((xmax-xmin, ymax-ymin, tmax-tmin + (tshift // support_trange + 1) * support_trange), dtype=torch.bool)
# md = torch.zeros((xmax-xmin, ymax-ymin, tmax-tmin), dtype=torch.bool)
# mhblock = accumulate(chunkify(hb_sgrid_loc_on_min, mhd.shape))
hqblock = accumulate(chunkify(hb_sgrid_loc_on_min, mhd.shape))
mqblock = accumulate(chunkify(hb_sgrid_loc_on_min_tshift, (xmax-xmin, ymax-ymin, tmax-tmin)))
mhblock = Block(location=hqblock.location, data=hqblock.data>0)
# mhblock.data = mhblock.data > 0
mqblock.data = mqblock.data > 0
mqexpandblock = Block(data=mqblock.data.clone().detach(), location=mqblock.location.clone().detach())

for i in range(mqblock.shape[0]):
    for j in range(mqblock.shape[1]):
        for k in range(mqblock.shape[2]):
            if mqblock.data[0,i,j,k]:
                mqexpandblock.data[0,i-1:i+1, j-1:j+1, k-pre_n*support_trange:k+post_n*support_trange] = True
print(mqexpandblock.shape, mqexpandblock.nbatches)

torch.sum(mqexpandblock.data), mqexpandblock.location, mqblock.location

In [ ]:
# mhblock.location, mhblock.shape, tmin
# coarse_effqb_sgrid.location[:,-1].max(), coarse_effqb_sgrid.location[:,-1].min()

In [ ]:
nhits = torch.max(torch.sum(mhblock.data, dim=-1)) + 1
nhits

In [ ]:
# construct Ae, (b, p1, p2, nhits, t=nq+nfr-1)
# construct A0, (b, p1, p2, t, t)
thres = 5
adc_hold_delay_ticks = 15
csa_reset_ticks = 1
Ae = torch.zeros((1, mqblock.shape[0], mqblock.shape[1], nhits, mqblock.shape[2]+fres3x3.shape[-1]-1))
A0 = torch.tril(torch.ones((mqblock.shape[2]+fres3x3.shape[-1]-1, mqblock.shape[2]+fres3x3.shape[-1]-1)), diagonal=1).expand(
        1, mqblock.shape[0], mqblock.shape[1], mqblock.shape[2]+fres3x3.shape[-1]-1, mqblock.shape[2]+fres3x3.shape[-1]-1
    ).clone().detach()
hq = torch.zeros((1, mqblock.shape[0], mqblock.shape[1], nhits))
last_timestamp = torch.full((1, mqblock.shape[0], mqblock.shape[1], 1), fill_value=1E-9, dtype=torch.int)

start_timeindex = None
end_timeindex = None
for i in range(nhits):
    trange = torch.arange(mqblock.shape[2]+fres3x3.shape[-1]-1)
    print(trange.shape)
    timestamp_idx = torch.argmax(mhblock.data.to(torch.int), dim=-1, keepdim=True)
    src = torch.zeros_like(timestamp_idx, dtype=mhblock.data.dtype, device=mhblock.device)
    print(timestamp_idx.shape, mhblock.data.shape, timestamp_idx[0,0,0])
    ishit = torch.gather(mhblock.data, dim=-1, index=timestamp_idx)
    timestamp_idx[~ishit] = -1E9
    # print(ishit.shape, )
    if i == 0:
        Ae[..., i, :] = trange[None, None, None, None, :] + tmin <= (timestamp_idx - adc_hold_delay_ticks)
        hq[..., i] = thres
        start_timeindex = timestamp_idx
    if i == 1:
        Ae[..., i, :] = (
            (trange[None, None, None, None, :] <= (timestamp_idx))
            & (trange[None, None, None, None, :] > (timestamp_idx - adc_hold_delay_ticks))
         )
        print(ishit.shape, torch.clamp(timestamp_idx, min=0).shape, hqblock.data.shape, torch.gather(hqblock.data, dim=-1, index=
                         torch.clamp(timestamp_idx, min=0)).squeeze(-1).shape)
        hq[..., i] =  torch.where(
            ishit.squeeze(-1),
            torch.gather(hqblock.data, dim=-1, index=
                         torch.clamp(timestamp_idx, min=0)).squeeze(-1) - thres,
            0,
        )
        last_timestamp = timestamp_idx.clone().detach()
        mhblock.data.scatter_(dim=-1, index=torch.where(ishit, timestamp_idx, 0), src=src)
    if i > 1:
        Ae[..., i, :] = (trange[None, None, None, None, :] <=  timestamp_idx) & (
                        trange[None, None, None, None, :] >= (last_timestamp + csa_reset_ticks)
        )
        hq[..., i] =  torch.where(
            ishit.squeeze(-1),
            torch.gather(hqblock.data, dim=-1, index=
                         torch.clamp(timestamp_idx, min=0)).squeeze(-1),
            0,
        )
        last_timestamp = timestamp_idx.clone().detach()
        mhblock.data.scatter_(dim=-1, index=torch.where(ishit, timestamp_idx, 0), src=src)
    if i == nhits - 1:
        end_timeindex = timestamp_idx
# fill zeros to stard and end index
start_timeindex = torch.where(start_timeindex == -1E9, torch.zeros_like(start_timeindex), start_timeindex)
end_timeindex = torch.where(end_timeindex == -1E9, torch.zeros_like(end_timeindex), end_timeindex)
for i in range(A0.shape[0]):
    for j in range(A0.shape[1]):
        for k in range(A0.shape[2]):
            for l in range(A0.shape[3]):
                A0[i,j,k,l,start_timeindex[i,j,k]:end_timeindex[i,j,k]] = 0



In [ ]:
X_hat, iterations =  solve_nonnegative_3d(Ae, A0, torch.tensor(fres3x3), hq, mqexpandblock.data, lam_l1=0.1, lam_dx=0.1, lam_a0=0.1)